Vamos ahora a tratar las imputaciones registradas con el sistema MES

En las imputaciones es donde esta el detalle real de ejecución (qué máquina, qué tiempos reales, quién y cuándo).

In [2]:
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [3]:
ruta_imputations = os.path.join("res\\raw", "imputaciones.csv")
print(ruta_imputations)

res\raw\imputaciones.csv


In [4]:
columns_imputations = ["FechaCreacionDte",	"smNodoId",	"smNodoNme",	"smOrdenId",	"smMaterialId",	"smOrdenNme",	"smFaseId",	"smFaseNme",	"CantidadManualOKNbr",	"CantidadManualNOKNbr",	"MinutosRealNbr",	"MinutosPreparacionNbr",	"MinutosProduccionNbr",	"PorcentajeDedicacionNbr",	"ProductivoInd",	"smIncidenciaId",	"smIncidenciaNme",	"smFasePersonalizado1Txt",	"smNodoId2",	"smNodoPersonalizado1Nbr",	"smNodoPersonalizado2Nbr",	"ssEnumFaseEstadoId",	"SmFaseUid",	"TraspasadoInd",	"JsonBodyTxt",	"JsonResponseTxt"]

pd.set_option("display.max_columns", None)  
pd.set_option("display.width", None)        

df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)

C:\Users\jaume\AppData\Local\Temp\ipykernel_7996\1223014052.py:6: DtypeWarning: Columns (0: smFaseId, 1: smIncidenciaId, 2: smIncidenciaNme) have mixed types. Specify dtype option on import or set low_memory=False.
  df_imputations = pd.read_csv(ruta_imputations, sep=";", header=None, names=columns_imputations)


In [5]:
display(df_imputations.head(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseId,smFaseNme,CantidadManualOKNbr,CantidadManualNOKNbr,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,PorcentajeDedicacionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,smFasePersonalizado1Txt,smNodoId2,smNodoPersonalizado1Nbr,smNodoPersonalizado2Nbr,ssEnumFaseEstadoId,SmFaseUid,TraspasadoInd,JsonBodyTxt,JsonResponseTxt
0,2026-08-05 00:57:09.823,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2606156,PE1JNCX0012,Portacápsulas F704501049704(150713),30,TPP,0.0,0.0,0.0,0.0,0.0,0.0,1,NaN,NaN,NaN,TN09,NaN,NaN,INICIADA_OLANET,C0DF4C4C-2E19-483E-B9CD-6506FC7BABA0,1,"{""CodCompany"":""100"",""CodSite"":""01"",""Imputation...","{""Result"":true,""errorDetails"":[],""newIDImputat..."


In [6]:
display(df_imputations.shape)

(196576, 26)

## Limpieza de atributos
Eliminar del dataset aquellos atributos prescindibles. 


In [7]:
def eliminar_columnas(df):
    """
    Elimina de df_imputations las columnas marcadas como innecesarias.
    Devuelve un DataFrame nuevo (no modifica el original).
    """

    cols_sobran = [
        "JsonResponseTxt", "JsonBodyTxt", "TraspasadoInd", "ssEnumFaseEstadoId", 
        "smNodoPersonalizado2Nbr", "smNodoPersonalizado1Nbr", "smNodoId2",
        "smFasePersonalizado1Txt", "PorcentajeDedicacionNbr", "CantidadManualOKNbr",
        "CantidadManualNOKNbr", "smFaseId",
        
    ]

    cols_a_quitar = [c for c in cols_sobran if c in columns_imputations]
    df_out = df.drop(columns=cols_a_quitar)

    # Aviso por si alguna de la lista no se encontró (typo o nombre distinto)
    no_encontradas = [c for c in cols_sobran if c not in df_imputations.columns]
    if no_encontradas:
        print(f"Aviso: {len(no_encontradas)} columnas de la lista no estaban en df: {no_encontradas}")

    print(f"Eliminadas {len(cols_a_quitar)} columnas. "
          f"Quedan {df_out.shape[1]} columnas, {df_out.shape[0]} filas.")
    return df_out

In [8]:
df_imputations_limpieza = eliminar_columnas(df_imputations)

Eliminadas 12 columnas. Quedan 14 columnas, 196576 filas.


In [9]:
display(df_imputations_limpieza.head(1))

,FechaCreacionDte,smNodoId,smNodoNme,smOrdenId,smMaterialId,smOrdenNme,smFaseNme,MinutosRealNbr,MinutosPreparacionNbr,MinutosProduccionNbr,ProductivoInd,smIncidenciaId,smIncidenciaNme,SmFaseUid
0,2026-08-05 00:57:09.823,TN09,TPP DOOSAN PUMA 240 (TPP) Torn C.N.C,OF2606156,PE1JNCX0012,Portacápsulas F704501049704(150713),TPP,0.0,0.0,0.0,1,NaN,NaN,C0DF4C4C-2E19-483E-B9CD-6506FC7BABA0


### Limpieza por intervalo de fechas
Trabajamos dentro del rango entre junio 2025 y antes de agosto 2026.

In [10]:
def filtrar_desde_fecha(df, fecha_desde, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea >= fecha_desde.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro >= {fecha_desde}.")

    limite = pd.to_datetime(fecha_desde)
    df_out = df_out[df_out[columna] >= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (desde {fecha_desde}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [11]:
df_imputations_limpieza = filtrar_desde_fecha(df_imputations_limpieza, "2025-06-01")

Partida: 196576 fases
Conservadas (desde 2025-06-01): 141329 (descartadas 55247)


In [12]:
def filtrar_hasta_fecha(df, fecha_hasta, columna='FechaCreacionDte'):
    """Conserva solo las instancaias cuya fecha en 'columna' sea =< fecha_hasta.
    Convierte la columna a datetime por seguridad (sirve si viene como texto).
    Devuelve un DataFrame nuevo."""

    n_inicial = df.shape[0]
    df_out = df.copy()

    # Convertir a datetime de forma robusta. errors='coerce' -> fechas no
    # parseables se vuelven NaT (nulo temporal) en vez de reventar.
    df_out[columna] = pd.to_datetime(df_out[columna], errors='coerce')

    # Avisar si alguna fecha no se pudo interpretar
    n_nat = df_out[columna].isna().sum()
    if n_nat:
        print(f"Aviso: {n_nat} filas con fecha no interpretable (NaT). "
              f"Quedan fuera del filtro =< {fecha_hasta}.")

    limite = pd.to_datetime(fecha_hasta)
    df_out = df_out[df_out[columna] <= limite].copy()

    print(f"Partida: {n_inicial} fases")
    print(f"Conservadas (hasta {fecha_hasta}): {df_out.shape[0]} "
          f"(descartadas {n_inicial - df_out.shape[0]})")
    return df_out

In [13]:
df_imputations_limpieza = filtrar_hasta_fecha(df_imputations_limpieza, "2026-08-01")

Partida: 141329 fases
Conservadas (hasta 2026-08-01): 140485 (descartadas 844)


Voy a guarda la tabla de las imputaciones en `res`

In [39]:
import pyarrow as pa

#df_imputations_limpieza["smFaseId"] = df_imputations_limpieza["smFaseId"].astype(str)

df_imputations_limpieza.to_parquet("res/procesed/df_imputations_limpieza.parquet")

## Filtrar imputaciones por maquinas productivas
Todas las imputaciones que no pertenezcan a la selección de maquinas productivas (`grupo_maquinas`) debo hacerlas desaparecer ya que son imputaciones a no emplear para mi proyecto. 

In [14]:
# cargo el df de maquinas y grupos
maquina_grupo = pd.read_parquet("res//procesed/maquina_grupo.parquet")
display(maquina_grupo.head(1))

#print(len(maquina_grupo["CodigoMaquina"].unique()))
#print(len(maquina_grupo["CodigoMaquina"]))

,IDMaquina,CodigoMaquina,NombreMaquina,IDGrupo,CodigoGrupo,NombreGrupo
0,01672179-5f64-497a-a94b-5005cd1dd366,TL01,T.IBARNIA 50 CA Taladre,ef7fa531-d402-44b6-9cc6-693ca3d08677,MA,Màquines Auxiliars


> Para comprender mejor la siguiente celda del código, quiero remarcar que para `maquinas_validas[CodigoMaquina]` tiene el mismo valor en `df_imputations_limpieza[smNodoId]`. Esto permite relacionar facilmente ambas tamblas mediante la máquina. 

In [15]:
maquinas_validas = maquina_grupo["CodigoMaquina"]

print(f"Imputaciones antes: {len(df_imputations_limpieza)}")
df_imput_productivas = df_imputations_limpieza[
    df_imputations_limpieza["smNodoId"].isin(maquinas_validas)
]
print(f"Imputaciones después: {len(df_imput_productivas)}")
print(f"Eliminadas (imputaciones en máquinas no productivas): {len(df_imputations_limpieza) - len(df_imput_productivas)}")

Imputaciones antes: 140485
Imputaciones después: 83457
Eliminadas (imputaciones en máquinas no productivas): 57028


> Aqui podria justificar que el 41% del total de las imputaciones desaparecen.

## Resolver fases imputadas en N nodos
Interesa averiguar si se ha dado una misma fase en distintas maquinas, por ello debo verificar si todas las imputaciones de una misma fase se han hecho en la misma máquina.   
Para cada imputación podemos saber en que nodo se ha realizado, un nodo puede ser una maquina o un centro de trabajo.

In [20]:
maquinas_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()
print(f"Total de fases productivas: {len(df_imput_productivas.groupby('SmFaseUid'))}")
print("")
print("Entre las fases productivas, el número de máquinas distintas por fase es:")
print(maquinas_por_fase.value_counts())

Total de fases productivas: 28985

Entre las fases productivas, el número de máquinas distintas por fase es:
smNodoId
1    27840
2     1083
3       56
4        4
5        2
Name: count, dtype: int64


Averiguar que maquinas son las que presentan imputaciones de una misma fase en distintos nodos es innecesario, lo que realmente me interesa es ver si este cambio de nodo traspasa entre distintos grupos de máquinas:

In [31]:
# recuento de nodos por fase (sobre imputaciones productivas)
nodos_por_fase = df_imput_productivas.groupby("SmFaseUid")["smNodoId"].nunique()

fases_mono = nodos_por_fase[nodos_por_fase == 1].index
fases_multi = nodos_por_fase[nodos_por_fase > 1].index

# --- 1. Peso en número de fases ---
total_fases = len(nodos_por_fase)
n_multi = len(fases_multi)
print(f"Fases totales:            {total_fases}")
print(f"Fases mono-nodo:          {len(fases_mono)} ({len(fases_mono)/total_fases:.1%})")
print(f"Fases multi-nodo:         {n_multi} ({n_multi/total_fases:.1%})")

# --- 2. Peso en tiempo real (¿son marginales también en volumen de trabajo?) ---
tiempo_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]["MinutosRealNbr"].sum()
tiempo_total = df_imput_productivas["MinutosRealNbr"].sum()
print(f"\nTiempo real en fases multi-nodo: {tiempo_multi/tiempo_total:.1%} del total")

Fases totales:            28985
Fases mono-nodo:          27840 (96.0%)
Fases multi-nodo:         1145 (4.0%)

Tiempo real en fases multi-nodo: 14.1% del total


pocas fases (4%) pero con peso desproporcionado en tiempo (14%). No son ruido marginal; son fases más "pesadas" que la media.
el 4% de las fases acumula el 14% del tiempo real. Eso significa que una fase multi-nodo tiene, de media, unas 3-4 veces más tiempo real que una fase normal. Tiene sentido lógico — son fases largas o complejas, y precisamente por eso se repartieron entre varias máquinas o sesiones.

Voy a decidir conservalas, teniendolas todas bajo una máquina dominante, es decir, la maquina que haya agrupado mayor tiempo. 

¿En que máquinas aparecen las fases multi nodo?

In [32]:
# ¿en qué máquinas aparecen las fases multi-nodo?
imput_multi = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_multi)]

t = imput_multi.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()
fraccion_dominante = t.groupby("SmFaseUid").max() / t.groupby("SmFaseUid").sum()
print(f"Fases donde la dominante concentra >80% del tiempo: {(fraccion_dominante > 0.8).mean():.1%}")

Fases donde la dominante concentra >80% del tiempo: 36.8%


En el 36,8% de las fases multi-nodo la máquina dominante concentra más del 80% del tiempo, es decir, en el 63,2% de estas fases, el tiempo está genuinamente repartido la "dominante" no domina tanto, convive con otras máquinas con peso real.  
  
Voy a trabajar con ese 36,8% de fases donde es facilmente identificable la maquina dominante, para las 63,2% restantes voy a prescindir, obviando las mismas. 

In [33]:
# tiempo por (fase, nodo)
tiempo_fase_nodo = df_imput_productivas.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum()

# fracción de la dominante en cada fase
dominante = tiempo_fase_nodo.groupby("SmFaseUid").max()
total = tiempo_fase_nodo.groupby("SmFaseUid").sum()
fraccion_dominante = dominante / total

# fases válidas: la dominante concentra >80% (esto incluye automáticamente las mono-nodo, que son 100%)
fases_validas = fraccion_dominante[fraccion_dominante > 0.8].index

print(f"Fases totales antes:     {fraccion_dominante.shape[0]}")
print(f"Fases válidas (dominante >80%): {len(fases_validas)}")
print(f"Fases descartadas (repartidas): {fraccion_dominante.shape[0] - len(fases_validas)}")

Fases totales antes:     28985
Fases válidas (dominante >80%): 28258
Fases descartadas (repartidas): 727


Entonces partimos de 140.485 imputaciones, el recorte a operaciones productivas elimina 57.028 (~41%), dejando 83.457 imputaciones repartidas en ~28.985 fases productivas, de las cuales nos quedamos con 28258 donde podemos identificar una maquina dominante como la responsable de la realización del trabajo en la fase. 

## Tabla final imputaciones: una instancia por cada fase
Finalmente voy a construir la tabla final, con una fila por fase válida, con su máquina dominante y sus tiempos reales agregados.

In [ ]:
# 1. me quedo solo con las imputaciones de las fases válidas
imput_validas = df_imput_productivas[df_imput_productivas["SmFaseUid"].isin(fases_validas)]

# 2. máquina dominante de cada fase (el nodo con más tiempo real)
    # Agrupa por la pareja fase + nodo, y suma el tiempo real de cada combinación.
tiempo_fase_nodo = imput_validas.groupby(["SmFaseUid", "smNodoId"])["MinutosRealNbr"].sum().reset_index()
    # Encontrar, por fase, la fila del tiempo máximo:
idx_dom = tiempo_fase_nodo.groupby("SmFaseUid")["MinutosRealNbr"].idxmax()
    # Recuperar esas filas y quedarse con fase + nodo
dominante = tiempo_fase_nodo.loc[idx_dom, ["SmFaseUid", "smNodoId"]].rename(columns={"smNodoId": "MaquinaDominante"})

# 3. tiempos agregados por fase (suma de TODAS sus imputaciones)
tiempos = imput_validas.groupby("SmFaseUid").agg(
    smOrdenId=("smOrdenId", "first"),
    smMaterialId=("smMaterialId", "first"),
    smFaseNme=("smFaseNme", "first"),
    smOrdenNme=("smOrdenNme", "first"),
    MinutosReal_total=("MinutosRealNbr", "sum"),
    MinutosPreparacion_total=("MinutosPreparacionNbr", "sum"),
    MinutosProduccion_total=("MinutosProduccionNbr", "sum"),
    n_imputaciones=("MinutosRealNbr", "size"),
).reset_index()

# 4. uno máquina dominante + tiempos en la tabla final (1 fila por fase)
fase_agregada = dominante.merge(tiempos, on="SmFaseUid", how="left")

print(f"Fases en la tabla final: {len(fase_agregada)}")
display(fase_agregada.head())

Fases en la tabla final: 28258


,SmFaseUid,MaquinaDominante,smOrdenId,smMaterialId,smFaseNme,smOrdenNme,MinutosReal_total,MinutosPreparacion_total,MinutosProduccion_total,n_imputaciones
0,0000245C-0BA3-42CD-9ABF-39C5B854CE59,ML01,OF2512622,PE2XA2X0105,MAR,Allonge coudée 103058945,0.78333,0.00000,0.78333,2
1,00013C31-3895-4370-BB3E-4255371E8D8F,TN09,OF2509545,PE1JNCX0012,TPP,Portacápsulas F704501049704(150713),41.76667,41.76667,0.00000,2
2,0001C0A9-BCA7-4F6C-8BB2-B36EC00F2594,MC02,OF2508408,PE1JNCX0029,CON,Portaelectrodes 1159D358819,24.26666,0.00000,64.46667,4
3,000542EB-1BC1-4F09-8AD8-D892596A91A5,TN10,OF2510311,ELEXA2X0340,TMO,ELECTRODO SUP PERNO 097625004020,41.31667,29.31667,12.00000,2
4,0007E37D-C243-4B8B-9372-3E596C45CB70,TN05,OF2600009,BR0XA2X0058,TPG-CNY,Sonderpinole 5658644,12.33334,0.13334,12.20000,4


Esta es mi tabla de imputaciones, en total tenemos 28258 fases productivas imputadas, lo guardo en el repositorio. 

In [43]:
df_fases_MES = fase_agregada

df_fases_MES.to_parquet("res//procesed/df_fases_MES.parquet")